In [97]:
# Week 2: BM25 Retrieval on MS MARCO

In [98]:
!pip install rank-bm25

In [99]:
import numpy as np
from datasets import load_dataset
from rank_bm25 import BM25Okapi

In [100]:
dataset = load_dataset("ms_marco", "v2.1")
train_data = dataset["train"].select(range(2000))

In [101]:
print(train_data)
print("Number of samples:", len(train_data))

Dataset({
    features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
    num_rows: 2000
})
Number of samples: 2000


In [102]:
sample = train_data[0]
sample

{'answers': ['The immediate impact of the success of the manhattan project was the only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.'],
 'passages': {'is_selected': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  'passage_text': ['The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.',
   'The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science.',
   'Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an atomic bomb possible. The success of th

In [103]:
print("Query:", sample["query"])
print("Number of passages:", len(sample["passages"]["passage_text"]))
print("First passage:", sample["passages"]["passage_text"][0])

Query: )what was the immediate impact of the success of the manhattan project?
Number of passages: 10
First passage: The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.


In [104]:
corpus = []

for sample in train_data:
    passages = sample["passages"]["passage_text"]
    for p in passages:
        if p and p.strip():
            corpus.append(p)

print("Corpus size:", len(corpus))
print("First passage:", corpus[0])

Corpus size: 19955
First passage: The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.


In [105]:
tokenized_corpus = [doc.lower().split() for doc in corpus]
print(tokenized_corpus[0][:20])

['the', 'presence', 'of', 'communication', 'amid', 'scientific', 'minds', 'was', 'equally', 'important', 'to', 'the', 'success', 'of', 'the', 'manhattan', 'project', 'as', 'scientific', 'intellect']


In [106]:
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index built successfully.")

BM25 index built successfully.


In [107]:
query = "what is the capital of france"
tokenized_query = query.lower().split()

scores = bm25.get_scores(tokenized_query)

print("Number of scores:", len(scores))
print("Max score:", np.max(scores))

Number of scores: 19955
Max score: 20.889330125332485


In [108]:
top_k = np.argsort(scores)[::-1][:5]

for rank, idx in enumerate(top_k, start=1):
    print(f"Rank {rank}")
    print("Score:", scores[idx])
    print("Passage:", corpus[idx])
    print("-" * 80)

Rank 1
Score: 20.889330125332485
Passage: The most important crop was wheat. Wheat was used to make flour. The farmers ground the wheat grains in the seigneur's flour mill. They would pay the seigneur with part of the … flour for the use of his mill. France may be a small country in Western Europe, but the impact of French cu…. 2  The Weather in France Though France is only the size of Texas, it is situated in an area where weather varies greatly.
--------------------------------------------------------------------------------
Rank 2
Score: 20.366722506006482
Passage: Image: © Briangotts. Satellite view is showing Valletta, the capital city of Malta. Valletta is located in the central-eastern portion of the island of Malta at the Mediterranean coast.The city is colloquially known as Il-Belt (The City) in Maltese and is the island's principal cultural center.mage: © Briangotts. Satellite view is showing Valletta, the capital city of Malta. Valletta is located in the central-eastern port

In [109]:
real_query = train_data[0]["query"]
print("Query:", real_query)

Query: )what was the immediate impact of the success of the manhattan project?


In [110]:
tokenized_query = real_query.lower().split()
scores = bm25.get_scores(tokenized_query)
top_k = np.argsort(scores)[::-1][:5]

for rank, idx in enumerate(top_k, start=1):
    print(f"Rank {rank}")
    print("Score:", scores[idx])
    print("Passage:", corpus[idx])
    print("-" * 80)

Rank 1
Score: 41.849561038132066
Passage: The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.
--------------------------------------------------------------------------------
Rank 2
Score: 40.83984823799754
Passage: Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an atomic bomb possible. The success of this project would forever change the world forever making it known that something this powerful can be manmade.
--------------------------------------------------------------------------------
Rank 3
Score: 35.74696034827184
Passage: Manhattan Project. The Manhattan Project was a research and development undertaking during World War II that produced the first nuclear w

In [111]:
def retrieve(query, bm25, corpus, k=5):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_k = np.argsort(scores)[::-1][:k]

    results = []
    for idx in top_k:
        results.append({
            "score": float(scores[idx]),
            "passage": corpus[idx]
        })
    return results

In [112]:
results = retrieve("what is the capital of france", bm25, corpus, k=5)

for i, r in enumerate(results, start=1):
    print(f"Rank {i}")
    print("Score:", r["score"])
    print("Passage:", r["passage"])
    print("-" * 80)

Rank 1
Score: 20.889330125332485
Passage: The most important crop was wheat. Wheat was used to make flour. The farmers ground the wheat grains in the seigneur's flour mill. They would pay the seigneur with part of the … flour for the use of his mill. France may be a small country in Western Europe, but the impact of French cu…. 2  The Weather in France Though France is only the size of Texas, it is situated in an area where weather varies greatly.
--------------------------------------------------------------------------------
Rank 2
Score: 20.366722506006482
Passage: Image: © Briangotts. Satellite view is showing Valletta, the capital city of Malta. Valletta is located in the central-eastern portion of the island of Malta at the Mediterranean coast.The city is colloquially known as Il-Belt (The City) in Maltese and is the island's principal cultural center.mage: © Briangotts. Satellite view is showing Valletta, the capital city of Malta. Valletta is located in the central-eastern port

In [113]:
import sys
sys.path.append("..")

from src.bm25_retriever import BM25Retriever

In [114]:
retriever = BM25Retriever(corpus)
results = retriever.retrieve("what is the capital of france", k=5)

for i, r in enumerate(results, start=1):
    print(f"Rank {i}")
    print("Score:", r["score"])
    print("Passage:", r["passage"])
    print("-" * 80)

Rank 1
Score: 20.889330125332485
Passage: The most important crop was wheat. Wheat was used to make flour. The farmers ground the wheat grains in the seigneur's flour mill. They would pay the seigneur with part of the … flour for the use of his mill. France may be a small country in Western Europe, but the impact of French cu…. 2  The Weather in France Though France is only the size of Texas, it is situated in an area where weather varies greatly.
--------------------------------------------------------------------------------
Rank 2
Score: 20.366722506006482
Passage: Image: © Briangotts. Satellite view is showing Valletta, the capital city of Malta. Valletta is located in the central-eastern portion of the island of Malta at the Mediterranean coast.The city is colloquially known as Il-Belt (The City) in Maltese and is the island's principal cultural center.mage: © Briangotts. Satellite view is showing Valletta, the capital city of Malta. Valletta is located in the central-eastern port

In [116]:
eval_queries = dataset["validation"].select(range(200))

In [117]:
passage_labels = []

for sample in train_data:

    labels = sample["passages"]["is_selected"]

    for l in labels:
        passage_labels.append(l)

In [118]:
def compute_mrr(bm25, queries, corpus, labels, k=10):

    reciprocal_ranks = []

    for sample in queries:

        query = sample["query"]
        tokenized_query = query.lower().split()

        scores = bm25.get_scores(tokenized_query)

        ranked = np.argsort(scores)[::-1][:k]

        rr = 0

        for rank, idx in enumerate(ranked, start=1):

            if labels[idx] == 1:
                rr = 1 / rank
                break

        reciprocal_ranks.append(rr)

    return np.mean(reciprocal_ranks)

In [119]:
mrr10 = compute_mrr(
    bm25,
    train_data.select(range(200)),
    corpus,
    passage_labels,
    k=10
)

print("MRR@10:", mrr10)

MRR@10: 0.27162698412698416


In [120]:
print("================================")
print("BM25 Retrieval Evaluation")
print("Corpus size:", len(corpus))
print("Queries evaluated:", 200)
print("MRR@10:", round(mrr10,4))
print("================================")

BM25 Retrieval Evaluation
Corpus size: 19955
Queries evaluated: 200
MRR@10: 0.2716
